# Tome 2, Chapter 1: Groups as Symmetries (Python companion)

Exercises first, solutions at the end. We build permutations, multiplication tables and the rotation group of the cube
from scratch, test Lagrange's theorem by brute force, and meet a first computer-science thread: **sorting by adjacent swaps
is a walk in a Cayley graph of $\mathfrak S_n$**, and bubble sort's worst case is the diameter of that graph.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import math
from collections import Counter, deque
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

## Exercises

**P1.1 (★) Permutations.** Represent a permutation of $\{0,\dots,n-1\}$ as a tuple `p` with `p[i]` the image of `i`. Write `compose(p, q)` (first `q`, then `p`), `inverse(p)`, `cycles(p)` and `order(p)`. Then tabulate how many elements of $\mathfrak S_5$ have each order. *Guess first: which orders occur, and which is most common?*

In [ ]:
# Your code here


**P1.2 (★) Multiplication tables.** Build the table of $D_n$ as permutations of the vertices (Proposition 1.3), and of $(\Z/n\Z)^*$. Check that every row and column is a permutation (Proposition 1.2(d)), and draw the table of $D_6$ as a coloured grid, like Figure 1.1.

In [ ]:
# Your code here


**P1.3 (★★) Lagrange by brute force.** List all subgroups of $\mathfrak S_4$ (every subgroup of $\mathfrak S_4$ is generated by at most two elements, which you may take for granted here). Print their orders. Which divisors of $24$ occur, and how many subgroups has each order? Is there a subgroup of order $6$ inside $A_4$?

In [ ]:
# Your code here


**P1.4 (★★) The cube.** Generate the $48$ signed permutation matrices, keep those of determinant $1$, and classify the $24$ rotations by their trace (the trace of a rotation by angle $\theta$ is $1+2\cos\theta$). Then compute the permutation each rotation induces on the four long diagonals and check that you get all of $\mathfrak S_4$ (Exercise 1.6).

In [ ]:
# Your code here


**P1.5 (★★★) CS application: sorting in a Cayley graph.** The *Cayley graph* of $\mathfrak S_n$ for the generators $s_i=(i\ i{+}1)$ has the $n!$ permutations as vertices, with an edge from $\sigma$ to $\sigma s_i$. A sequence of adjacent swaps that sorts a list is a path to the identity. Compute by breadth-first search the distance from every permutation to the identity for $n\le7$, compare it with the number of inversions, and find the diameter. Then draw the graph for $n=4$.

*How to think about it:* bubble sort performs exactly one adjacent swap per inversion, and no algorithm restricted to adjacent swaps can do better, because one swap changes the number of inversions by exactly one.

In [ ]:
# Your code here


---
## Solutions

### P1.1

In [ ]:
def compose(p, q):
    return tuple(p[q[i]] for i in range(len(q)))

def inverse(p):
    inv = [0] * len(p)
    for i, x in enumerate(p):
        inv[x] = i
    return tuple(inv)

def cycles(p):
    seen, out = set(), []
    for i in range(len(p)):
        if i not in seen:
            c, j = [], i
            while j not in seen:
                seen.add(j); c.append(j); j = p[j]
            if len(c) > 1:
                out.append(tuple(c))
    return out

def order(p):
    return math.lcm(*[len(c) for c in cycles(p)]) if cycles(p) else 1

p = (1, 2, 0, 4, 3)
print("p =", p, " cycles:", cycles(p), " order:", order(p), " p * p^-1 =", compose(p, inverse(p)))
counts = Counter(order(p) for p in itertools.permutations(range(5)))
print("orders in S_5:", dict(sorted(counts.items())), " total", sum(counts.values()))

The orders are $1,2,3,4,5,6$. The most common is $4$ ($30$ four-cycles), then $2$ ($10$ transpositions and $15$ double transpositions), then $5$ ($24$ five-cycles). Order $6$ comes from a $3$-cycle times a disjoint transposition ($20$ elements). The order of a permutation is the lcm of its cycle lengths, which is why no element of $\mathfrak S_5$ has order $7$, $8$ or $10$.

### P1.2

In [ ]:
def dihedral(n):
    r = tuple((i + 1) % n for i in range(n))
    s = tuple((-i) % n for i in range(n))
    e = tuple(range(n))
    rot = [e]
    for _ in range(n - 1):
        rot.append(compose(r, rot[-1]))
    return rot + [compose(s, x) for x in rot]

def table(elems, mul):
    idx = {g: i for i, g in enumerate(elems)}
    return np.array([[idx[mul(a, b)] for b in elems] for a in elems])

def is_latin(T):
    n = len(T)
    return all(sorted(row) == list(range(n)) for row in T) and all(sorted(col) == list(range(n)) for col in T.T)

D6 = dihedral(6)
T = table(D6, compose)
print("D_6 has", len(D6), "elements; Latin square:", is_latin(T))
U = [a for a in range(1, 15) if math.gcd(a, 15) == 1]
TU = table(U, lambda a, b: a * b % 15)
print("(Z/15)^* =", U, "; Latin square:", is_latin(TU))

fig, ax = plt.subplots(1, 2, figsize=(10, 4.6))
ax[0].imshow(T, cmap="tab20"); ax[0].set_title("$D_6$: rotations first, then reflections")
ax[1].imshow(TU, cmap="tab10"); ax[1].set_title("$(\\mathbb{Z}/15)^*$")
ax[1].set_xticks(range(len(U)), U); ax[1].set_yticks(range(len(U)), U)
plt.show()

The table of $D_6$ splits into four $6\times6$ blocks, as in Figure 1.1: rotation times rotation and reflection times reflection give rotations. The table of the abelian group $(\Z/15\Z)^*$ is symmetric about its diagonal; that of $D_6$ is not.

### P1.3

In [ ]:
def generated(gens, n):
    e = tuple(range(n))
    H, frontier = {e}, [e]
    while frontier:
        new = []
        for h in frontier:
            for g in gens:
                x = compose(g, h)
                if x not in H:
                    H.add(x); new.append(x)
        frontier = new
    return frozenset(H)

S4 = list(itertools.permutations(range(4)))
subgroups = {generated([a, b], 4) for a in S4 for b in S4}
by_order = Counter(len(H) for H in subgroups)
print(len(subgroups), "subgroups; by order:", dict(sorted(by_order.items())))
assert all(24 % k == 0 for k in by_order)

def sign(p):
    return (-1) ** sum(1 for i in range(len(p)) for j in range(i + 1, len(p)) if p[i] > p[j])
A4 = frozenset(p for p in S4 if sign(p) == 1)
print("subgroups of order 6 inside A_4:", sum(1 for H in subgroups if len(H) == 6 and H <= A4))

$\mathfrak S_4$ has $30$ subgroups, of every order dividing $24$ (orders $1,2,3,4,6,8,12,24$). The four subgroups of order $6$ are copies of $\mathfrak S_3$ (the stabilisers of a point) and none lies in $A_4$: Lagrange's theorem has no converse, as announced in the chapter and proved in Chapter 4. (That two generators suffice for every subgroup of $\mathfrak S_4$ is checked here only implicitly; the counts agree with the classical list.)

### P1.4

In [ ]:
rotations = []
for perm in itertools.permutations(range(3)):
    for signs in itertools.product([1, -1], repeat=3):
        M = np.zeros((3, 3), dtype=int)
        for j in range(3):
            M[perm[j], j] = signs[j]
        if round(np.linalg.det(M)) == 1:
            rotations.append(M)
print(len(rotations), "rotations")
kind = {3: "identity", -1: "180 deg (face or edge axis)", 1: "+-90 deg about a face axis", 0: "+-120 deg about a diagonal"}
print(Counter(kind[int(np.trace(M))] for M in rotations))

diagonals = [np.array(v) for v in [(1, 1, 1), (1, 1, -1), (1, -1, 1), (-1, 1, 1)]]
def on_diagonals(M):
    out = []
    for v in diagonals:
        w = M @ v
        out.append(next(i for i, d in enumerate(diagonals) if (w == d).all() or (w == -d).all()))
    return tuple(out)
images = {on_diagonals(M) for M in rotations}
print("distinct permutations of the diagonals:", len(images), "= all of S_4:", images == set(S4))

The traces give $1$ identity, $9$ half-turns ($3$ about face axes and $6$ about edge axes, both of trace $-1$), $6$ quarter-turns and $8$ third-turns: $1+9+6+8=24$, the count of Proposition 1.4. The $24$ rotations induce $24$ distinct permutations of the diagonals, so the map to $\mathfrak S_4$ is a bijection.

### P1.5

In [ ]:
def cayley_distances(n):
    e = tuple(range(n))
    gens = [tuple(list(range(i)) + [i + 1, i] + list(range(i + 2, n))) for i in range(n - 1)]
    dist, q = {e: 0}, deque([e])
    while q:
        x = q.popleft()
        for g in gens:
            y = compose(x, g)
            if y not in dist:
                dist[y] = dist[x] + 1; q.append(y)
    return dist, gens

def inversions(p):
    return sum(1 for i in range(len(p)) for j in range(i + 1, len(p)) if p[i] > p[j])

for n in range(2, 8):
    dist, _ = cayley_distances(n)
    same = all(d == inversions(p) for p, d in dist.items())
    print(f"n = {n}: {len(dist):5d} permutations, distance = inversions: {same}, diameter {max(dist.values())} = n(n-1)/2 = {n * (n - 1) // 2}")

dist, gens = cayley_distances(4)
G = nx.Graph()
for x in dist:
    for g in gens:
        G.add_edge(x, compose(x, g))
pos = nx.spring_layout(G, seed=3)
plt.figure(figsize=(6, 6))
nx.draw(G, pos, node_size=60, node_color=[dist[v] for v in G], cmap="viridis", edge_color="gray")
plt.title("Cayley graph of $\\mathfrak{S}_4$ for adjacent swaps (colour = distance to the identity)")
plt.show()

The distance to the identity is exactly the number of inversions, and the diameter is $\binom n2$, reached only by the reversed list. So bubble sort is optimal among algorithms that only swap neighbours, and any such algorithm needs $\binom n2$ swaps in the worst case. The drawing for $n=4$ is the *permutohedron*: $24$ vertices, squares and hexagons, the shape of the truncated octahedron.